In [1]:
from pypdf import PdfReader

extracting the raw text from the data

In [2]:
from pypdf import PdfReader

pdf_path = r"C:\Users\shubh\OneDrive\Desktop\learnign_RAG\sample_reports\WM17S.pdf"

reader = PdfReader(pdf_path)

report_text = ""

for i, page in enumerate(reader.pages):
    page_text = page.extract_text()

    report_text += f"\n\n===== PAGE {i+1} =====\n\n"
    report_text += page_text

print("Characters:", len(report_text))

Characters: 18170


In [3]:
print(report_text)



===== PAGE 1 =====

Report Status    
Male
25 Years:
:
:
:
Age
Gender
Reported        
P
7/11/2023  11:08:00AM
SELF
WM17SPF
Mr.   DUMMY:
:
:
:
:
Name        
Lab No.    
Ref By 
Collected       
A/c Status 
17/1/2024  10:55:48AM
Revised
Collected at            : Processed at             :LPL-ROHINI (NATIONAL REFERENCE LAB)
National Reference laboratory, Block E, Sector 
18, ROHINI
DELHI 110085
LPL-NATIONAL REFERENCE LAB
National Reference laboratory, Block E, 
Sector 18, Rohini, New Delhi -110085
Test Report 
Test Name Results Units Bio. Ref. Interval
SWASTHFIT SUPER 4
LIVER & KIDNEY PANEL, SERUM
1.00Creatinine
(Modified Jaffe,Kinetic)
 0.70 - 1.30 mg/dL
107GFR Estimated
(CKD EPI Equation 2021)
 >59 mL/min/1.73m2
G1GFR Category
(KDIGO Guideline 2012)
  
40.00Urea
(Urease UV)
 13.00 - 43.00 mg/dL
18.68Urea Nitrogen  Blood
(Calculated)
 6.00 - 20.00 mg/dL
19BUN/Creatinine Ratio
(Calculated)
  
7.00Uric Acid
(Uricase)
 3.50 - 7.20 mg/dL
30.0AST (SGOT)
(IFCC without P5P)
 15.00 - 40.00 U

got complete raw extracted text

now we will break the data in meaningful chunks

Rightnow my data is a mess, i want to put in a way such that, it can become a useful chunk of infos
i need to do manual splitting , for this particular pdf , i have headers to separate it.




In [4]:
import re

# List of known section headers
section_headers = [
    "LIVER & KIDNEY PANEL, SERUM",
    "LIPID SCREEN, SERUM",
    "Glucose Fasting",
    "VITAMIN B12; CYANOCOBALAMIN",
    "VITAMIN D, 25 - HYDROXY, SERUM",
    "THYROID PROFILE,TOTAL, SERUM",
    "HbA1c (GLYCOSYLATED HEMOGLOBIN), BLOOD",
    "COMPLETE BLOOD COUNT; CBC"
]

# Create regex pattern
pattern = "(" + "|".join(map(re.escape, section_headers)) + ")"

# Split while keeping section names
parts = re.split(pattern, report_text)

chunks = []

for i in range(1, len(parts), 2):
    section_name = parts[i].strip()

    if i + 1 < len(parts):
        section_content = parts[i + 1].strip()

        chunks.append({
            "section": section_name,
            "content": section_content
        })

print(f"Total Chunks: {len(chunks)}")

for chunk in chunks:
    print("\n" + "="*60)
    print("SECTION:", chunk["section"])
    print("="*60)
    print(chunk["content"][:500])

Total Chunks: 8

SECTION: LIVER & KIDNEY PANEL, SERUM
1.00Creatinine
(Modified Jaffe,Kinetic)
 0.70 - 1.30 mg/dL
107GFR Estimated
(CKD EPI Equation 2021)
 >59 mL/min/1.73m2
G1GFR Category
(KDIGO Guideline 2012)
  
40.00Urea
(Urease UV)
 13.00 - 43.00 mg/dL
18.68Urea Nitrogen  Blood
(Calculated)
 6.00 - 20.00 mg/dL
19BUN/Creatinine Ratio
(Calculated)
  
7.00Uric Acid
(Uricase)
 3.50 - 7.20 mg/dL
30.0AST (SGOT)
(IFCC without P5P)
 15.00 - 40.00 U/L
40.0ALT (SGPT)
(IFCC without P5P)
 10.00 - 49.00 U/L
50.0GGTP
(IFCC)
 0 - 73 U/L
100.00Alkaline Phospha

SECTION: LIPID SCREEN, SERUM
100.00Cholesterol, Total
(CHO-POD)
 <200.00 mg/dL
100.00Triglycerides
(GPO-POD)
 <150.00 mg/dL
30.00HDL  Cholesterol
(Enz Immunoinhibition)
 >40.00 mg/dL
50.00LDL Cholesterol, Calculated
(Calculated)
 <100.00 mg/dL
20.00VLDL Cholesterol,Calculated
(Calculated)
 <30.00 mg/dL
70Non-HDL Cholesterol
(Calculated)
 <130 mg/dL
Note
1. Measurements in the same patient can show physiological & analytical variations. Thre

In [5]:
from langchain_core.documents import Document

documents = []

for chunk in chunks:

    documents.append(
        Document(
            page_content=chunk["content"],
            metadata={
                "section": chunk["section"]
            }
        )
    )

print("Documents:", len(documents))

Documents: 8


In [6]:
print("Total Semantic Chunks:", len(chunks))

for c in chunks:
    print(c["section"])

Total Semantic Chunks: 8
LIVER & KIDNEY PANEL, SERUM
LIPID SCREEN, SERUM
Glucose Fasting
VITAMIN B12; CYANOCOBALAMIN
VITAMIN D, 25 - HYDROXY, SERUM
THYROID PROFILE,TOTAL, SERUM
HbA1c (GLYCOSYLATED HEMOGLOBIN), BLOOD
COMPLETE BLOOD COUNT; CBC


NOw converting my semantic chunks to langchain documents

In [7]:
from langchain_core.documents import Document

documents = []

for chunk in chunks:

    doc = Document(
        page_content=chunk["content"],
        metadata={
            "section": chunk["section"]
        }
    )

    documents.append(doc)

print("Documents created:", len(documents))

Documents created: 8


In [8]:
print(documents[0])

page_content='1.00Creatinine
(Modified Jaffe,Kinetic)
 0.70 - 1.30 mg/dL
107GFR Estimated
(CKD EPI Equation 2021)
 >59 mL/min/1.73m2
G1GFR Category
(KDIGO Guideline 2012)
  
40.00Urea
(Urease UV)
 13.00 - 43.00 mg/dL
18.68Urea Nitrogen  Blood
(Calculated)
 6.00 - 20.00 mg/dL
19BUN/Creatinine Ratio
(Calculated)
  
7.00Uric Acid
(Uricase)
 3.50 - 7.20 mg/dL
30.0AST (SGOT)
(IFCC without P5P)
 15.00 - 40.00 U/L
40.0ALT (SGPT)
(IFCC without P5P)
 10.00 - 49.00 U/L
50.0GGTP
(IFCC)
 0 - 73 U/L
100.00Alkaline Phosphatase (ALP)
(IFCC-AMP)
 30.00 - 120.00 U/L
1.00Bilirubin  Total
(Oxidation)
 0.30 - 1.20 mg/dL
0.20Bilirubin  Direct
(Oxidation)
 <0.3 mg/dL
0.80Bilirubin  Indirect
(Calculated)
 <1.10 mg/dL
8.00Total Protein
(Biuret)
 5.70 - 8.20 g/dL
4.00Albumin
(BCG)
 3.20 - 4.80 g/dL
1.00A : G Ratio
(Calculated)
 0.90 - 2.00 
4.00Globulin(Calculated)  2.0 - 3.5 gm/dL
9.00Calcium, Total
(Arsenazo III)
 8.70 - 10.40 mg/dL
*WM17SPF*
.
Page 1 of 7


===== PAGE 2 =====

Report Status    
Male
25 Year

since i don't have any API keys , so for emberdding and vecetorization ,  i will be using local running sentence-transofmer

In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6866.07it/s]


In [11]:
vector = embedding_model.encode(
    "What is my HbA1c?"
)

print(len(vector))

384


In [12]:
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings

C:\Users\shubh\AppData\Local\Temp\ipykernel_15608\1506071036.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [13]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

class LocalEmbeddings(Embeddings):

    def __init__(self):
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

In [14]:
embeddings = LocalEmbeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6436.82it/s]


In [15]:
db = FAISS.from_documents(
    documents,
    embeddings
)

In [16]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)

results = retriever.invoke(
    "What is my HbA1c?"
)

for doc in results:
    print(doc.metadata)
    print(doc.page_content[:300])

{'section': 'HbA1c (GLYCOSYLATED HEMOGLOBIN), BLOOD'}
(HPLC, NGSP certified)
HbA1c % 4.00 - 5.6010.0
Estimated average glucose (eAG) mg/dL240
Interpretation
HbA1c result is suggestive of  Diabetes/ Higher than glycemic goal in a known Diabetic patient. 
Please note, Glycemic goal should be individualized based on duration of diabetes, age/life expectan
{'section': 'Glucose Fasting'}
(Hexokinase)
 70 - 100 mg/dL
{'section': 'LIPID SCREEN, SERUM'}
100.00Cholesterol, Total
(CHO-POD)
 <200.00 mg/dL
100.00Triglycerides
(GPO-POD)
 <150.00 mg/dL
30.00HDL  Cholesterol
(Enz Immunoinhibition)
 >40.00 mg/dL
50.00LDL Cholesterol, Calculated
(Calculated)
 <100.00 mg/dL
20.00VLDL Cholesterol,Calculated
(Calculated)
 <30.00 mg/dL
70Non-HDL Cholesterol
(Ca


NO trying to use openrouter 


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="Put your key here",
    base_url="https://openrouter.ai/api/v1"
)

In [20]:
context = "\n\n".join(
    [doc.page_content for doc in results]
)

prompt = f"""
Use only the context below.

Context:
{context}

Question:
What are the problems i am having?

Answer:
"""

In [21]:
response = client.chat.completions.create(
    model="openrouter/auto",
    messages=[
        {"role":"user","content":prompt}
    ]
)

print(response.choices[0].message.content)

Based on the provided context, the problems you are having are:

*   **High HbA1c:** Your HbA1c result is 10.0%. The interpretation states this is "suggestive of Diabetes/ Higher than glycemic goal in a known Diabetic patient." The reference interval for non-diabetic adults is 4.0-5.6%.

*   **High Estimated Average Glucose (eAG):** Your estimated average glucose (eAG) is 240 mg/dL. While a specific reference range isn't given in the context for eAG, the high HbA1c strongly suggests this value is also elevated, indicating a problem with glucose control.
